In [1]:
import csv
import sys
import pickle as pkl
import pandas as pd
import random
import tqdm
import sklearn
import operator
from proposed_design_types import PROPOSED_DESIGN_TYPES

In [2]:
is_rct = lambda data: ['Randomized Controlled Trial' in ptyp_list for ptyp_list in data['ptyp']]
is_sr = lambda data: ['Systematic Review' in ptyp_list for ptyp_list in data['ptyp']]
is_casecontrol = lambda data: ['Case-Control Studies' in ptyp_list for ptyp_list in data['mesh']]
is_ct = lambda data: ['Clinical Trial' in ptyp_list for ptyp_list in data['ptyp']]
is_cohort = lambda data: ['Cohort Studies' in mesh_list for mesh_list in data['mesh']]
is_guideline = lambda data: ['Guideline' in ptyp_list for ptyp_list in data['ptyp']]
is_consensus = lambda data: ['Consensus' in ptyp_list for ptyp_list in data['mesh']]
is_qual = lambda data: ['Qualitative Research' in mesh_list for mesh_list in data['mesh']]
is_ctprotocol = lambda data: ['Clinical Trial Protocol' in ptyp_list for ptyp_list in data['ptyp']]

In [3]:
study_design_fns = ['is_rct',
'is_sr',
'is_casecontrol',
'is_ct',
'is_cohort',
'is_guideline',
'is_consensus',
'is_qual',
'is_ctprotocol']

In [4]:

filename_neg = 'pubmed_articles_neg.csv'

neg_data = pd.read_csv(
         filename_neg,
)

In [5]:
neg_data.columns

Index(['Unnamed: 0', 'pmid', 'status', 'indexing_method', 'title', 'abstract',
       'abstract_plaintext', 'authors', 'journal', 'journal_abbrv', 'year',
       'mesh', 'month', 'volume', 'issue', 'pages', 'ptyp', 'registry_ids',
       'dois'],
      dtype='object')

In [11]:
neg_data_human.head()

,Unnamed: 0,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
0,981,18223670,MEDLINE,Human,"Agonist binding, agonist affinity and agonist ...","[{'header': '_UNSTRUCTURED', 'text': 'Measurem...",Measurements of affinity and efficacy are fund...,"[{'Initials': 'PG', 'LastName': 'Strange', 'Fo...",British journal of pharmacology,Br J Pharmacol,2008.0,"['Binding Sites', 'Drug Agonism', 'Drug Design...",Apr,153,7,"{'page_from': '1353', 'page_to': '1363'}","['Journal Article', 'Review']",[],['10.1038/sj.bjp.0707672']
1,5415,18228324,MEDLINE,Human,Overview of protein phosphorylation.,"[{'header': '_UNSTRUCTURED', 'text': ""Phosphor...",Phosphorylation is the most common and importa...,"[{'Initials': 'BM', 'LastName': 'Sefton', 'For...",Current protocols in cell biology,Curr Protoc Cell Biol,2001.0,"['Animals', 'Humans', 'Phosphoprotein Phosphat...",May,Chapter 14,NaN,"{'page_from': 'Unit 14.1', 'page_to': 'Unit 14...","['Journal Article', 'Review']",[],['10.1002/0471143030.cb1401s00']
2,26636,18250622,MEDLINE,Human,[Management of sepsis of the biliary tract: in...,"[{'header': '_UNSTRUCTURED', 'text': ""Sepsis o...",Sepsis of the biliary tract is a severe diseas...,"[{'Initials': 'P', 'LastName': 'De Paolis', 'F...",Minerva gastroenterologica e dietologica,Minerva Gastroenterol Dietol,2002.0,"['Bile Duct Neoplasms', 'Cholangitis', 'Cholel...",Mar,48,1,"{'page_from': '37', 'page_to': '43'}","['English Abstract', 'Journal Article', 'Review']",[],[]
3,11613,18234712,MEDLINE,Human,"Signalling, inflammation and arthritis: NF-kap...","[{'header': '_UNSTRUCTURED', 'text': ""In the s...","In the synovial cells of patients with RA, act...","[{'Initials': 'RE', 'LastName': 'Simmonds', 'F...","Rheumatology (Oxford, England)",Rheumatology (Oxford),2008.0,"['Animals', 'Arthritis, Rheumatoid', 'Cell Lin...",May,47,5,"{'page_from': '584', 'page_to': '590'}","['Journal Article', 'Review']",[],['10.1093/rheumatology/kem298']
4,743,18223363,MEDLINE,Human,Active tuberculosis does not always imply the ...,[],NaN,"[{'Initials': 'A', 'LastName': 'Meybeck', 'For...",Journal of acquired immune deficiency syndrome...,J Acquir Immune Defic Syndr,2008.0,"['AIDS-Related Opportunistic Infections', 'Ant...",Feb,47,2,"{'page_from': '259', 'page_to': '261'}",['Letter'],[],['10.1097/QAI.0b013e318158bee5']


In [33]:
neg_data_human = neg_data[neg_data['indexing_method']== 'Human']

In [8]:
neg_data_human.columns

Index(['Unnamed: 0', 'pmid', 'status', 'indexing_method', 'title', 'abstract',
       'abstract_plaintext', 'authors', 'journal', 'journal_abbrv', 'year',
       'mesh', 'month', 'volume', 'issue', 'pages', 'ptyp', 'registry_ids',
       'dois'],
      dtype='object')

In [37]:
neg_data_human=neg_data_human.reset_index()

In [38]:
neg_data_human.to_csv('pubmed_articles_neg_human.csv')

# with open('pubmed_articles_neg_human.csv') as f_neg_human:
    

In [17]:
filename_pos_human = 'pubmed_articles_pos_human.csv'

# p = 0.2  # 20% of the lines
# keep the header, then take only 1% of lines
# if random from [0,1] interval is greater than 0.01 the row will be skipped
pos_data_human = pd.read_csv(
         filename_pos_human,
#          header=0, 
#          skiprows=lambda i: i>0 and random.random() > p
)

In [15]:
pos_data_human = pos_data_human.drop(columns='Unnamed: 0')

In [24]:
pos_data_human.to_csv('pubmed_articles_pos_human.csv')

In [39]:
prop_typ_id_dict = {}

for fn in study_design_fns:
    prop_typ_id_dict[fn] = {}
    print(fn)
    pos_data_human_typ = pos_data_human[eval(fn)(pos_data_human)]
    prop_typ_id_dict[fn]['pos'] = list(pos_data_human_typ.index)
    print('pos', len(prop_typ_id_dict[fn]['pos']))
    not_fn = lambda data:  list(map(operator.not_, eval(fn)(data)))
    neg_data_human_typ = neg_data_human[ not_fn(neg_data_human)]
    prop_typ_id_dict[fn]['neg'] = list(neg_data_human_typ.index)
    print('neg', len(prop_typ_id_dict[fn]['neg']))
    print('neg/pos', len(prop_typ_id_dict[fn]['neg'])/len(prop_typ_id_dict[fn]['pos']))

is_rct
pos 489295
neg 5585042
neg/pos 11.414467754626555
is_sr
pos 121063
neg 5659292
neg/pos 46.74666908964754
is_casecontrol
pos 65790
neg 5628219
neg/pos 85.54824441404469
is_ct
pos 593277
neg 5564942
neg/pos 9.38000630396931
is_cohort
pos 78688
neg 5632460
neg/pos 71.57965636437577
is_guideline
pos 33826
neg 5676411
neg/pos 167.81206764027672
is_consensus
pos 10093
neg 5678494
neg/pos 562.6170613296345
is_qual
pos 7577
neg 5673091
neg/pos 748.7252210637455
is_ctprotocol
pos 3235
neg 5682702
neg/pos 1756.6312210200927


In [40]:
import pickle as pkl

with open('pos_neg_dataset_ids.pkl', 'wb') as f:
    pkl.dump(prop_typ_id_dict, f)

In [16]:
neg_data_human_sr = neg_data_human[['Systematic Review' not in eval(ptyp) for ptyp in neg_data_human['ptyp']]]

In [61]:
human_sr_neg = human_sr_neg.drop(columns = ['indexing_method', 'abstract', 'month', 'volume', 'issue', 'dois', 'ptyp', 'registry_ids', 'pages', 'Unnamed: 0'])

In [62]:
human_sr_neg['label'] = 0

In [66]:
filename_pos = 'pubmed_articles_pos_human.csv'

# p = 0.2  # 20% of the lines
# keep the header, then take only 1% of lines
# if random from [0,1] interval is greater than 0.01 the row will be skipped
pos_data = pd.read_csv(
         filename_pos,
#          header=0, 
#          skiprows=lambda i: i>0 and random.random() > p
)

In [29]:
pos_data_human.drop(columns = ['Unnamed: 0', 'unnamed: 0'])

,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
0,18222599,MEDLINE,Human,Laparoscopic versus open partial nephrectomy: ...,"[{'header': 'OBJECTIVES', 'text': 'To critical...",OBJECTIVES\n\n\nTo critically review the curre...,"[{'Initials': 'F', 'LastName': 'Porpiglia', 'F...",European urology,Eur Urol,2008.0,"['Carcinoma, Renal Cell', 'Clinical Competence...",Apr,53,4,{},"['Journal Article', 'Review']",[],['10.1016/j.eururo.2008.01.025']
1,18222601,MEDLINE,Human,Editorial comment on: expression and prognosti...,[],NaN,"[{'Initials': 'P', 'LastName': 'Härkönen', 'Fo...",European urology,Eur Urol,2008.0,"['Annexin A3', 'Humans', 'Male', 'Prognosis', ...",Dec,54,6,"{'page_from': '1322', 'page_to': '1322'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.002']
2,18222602,MEDLINE,Human,Editorial comment on: celecoxib has potent ant...,[],NaN,"[{'Initials': 'S', 'LastName': 'Brandau', 'For...",European urology,Eur Urol,2008.0,"['Animals', 'BCG Vaccine', 'Carcinoma, Transit...",Sep,54,3,"{'page_from': '629', 'page_to': '630'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.014']
3,18222603,MEDLINE,Human,Editorial comment on: expression and prognosti...,[],NaN,"[{'Initials': 'GJ', 'LastName': 'van Leenders'...",European urology,Eur Urol,2008.0,"['Annexin A3', 'Humans', 'Male', 'Microarray A...",Dec,54,6,"{'page_from': '1323', 'page_to': '1323'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.003']
4,18222604,MEDLINE,Human,Considerations on the use of urine markers for...,[],NaN,"[{'Initials': 'BW', 'LastName': 'van Rhijn', '...",European urology,Eur Urol,2008.0,"['Biomarkers, Tumor', 'Carcinoma, Transitional...",May,53,5,"{'page_from': '880', 'page_to': '881'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.043']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8584786,24210460,MEDLINE,Human,Pregnancy-related knowledge and information ne...,"[{'header': '_UNSTRUCTURED', 'text': ""For wome...","For women with epilepsy (WWE), pregnancy is co...","[{'Initials': 'A', 'LastName': 'McGrath', 'For...",Epilepsy & behavior : E&B,Epilepsy Behav,2014.0,"['Anticonvulsants', 'Epilepsy', 'Female', 'Hea...",Feb,31,NaN,"{'page_from': '246', 'page_to': '255'}","['Journal Article', ""Research Support, Non-U.S...",[],['10.1016/j.yebeh.2013.09.044']
8584787,24210462,MEDLINE,Human,Australia's seizure divide - indigenous versus...,"[{'header': '_UNSTRUCTURED', 'text': 'Indigeno...",Indigenous Australians suffer the highest mort...,"[{'Initials': 'C', 'LastName': 'Plummer', 'For...",Epilepsy & behavior : E&B,Epilepsy Behav,2014.0,"['Adolescent', 'Adult', 'Australia', 'Ethnic G...",Feb,31,NaN,"{'page_from': '363', 'page_to': '368'}","['Comparative Study', 'Journal Article', ""Rese...",[],['10.1016/j.yebeh.2013.09.042']
8584788,24210464,MEDLINE,Human,Research on psychosocial aspects of epilepsy i...,"[{'header': '_UNSTRUCTURED', 'text': 'This stu...",This study reviewed research conducted on the ...,"[{'Initials': 'JM', 'LastName': 'Al-Khateeb', ...",Epilepsy & behavior : E&B,Epilepsy Behav,2014.0,"['Biomedical Research', 'Epilepsy', 'Health Kn...",Feb,31,NaN,"{'page_from': '256', 'page_to': '262'}","['Journal Article', 'Review']",[],['10.1016/j.yebeh.2013.09.033']
8584789,24210468,MEDLINE,Human,Emerging methods for the fabrication of polyme...,"[{'header': '_UNSTRUCTURED', 'text': 'Hollow p...",Hollow polymer capsules are attracting increas...,"[{'Initials': 'J', 'LastName': 'Cui', 'ForeNam...",Advances in colloid and interface science,Adv Colloid Interface Sci,2014.0,"['Capsules', 'Chemical Phenomena', 'Drug Compo...",May,207,NaN,"{'page_from': '14', 'page_to': '31'}","['Journal Article', ""Research Support, Non-U.S...",[],['10.1016/j.cis.2013.10.012']


In [68]:
human_sr_pos = pos_data[['Systematic Review' in eval(ptyp) for ptyp in pos_data['ptyp']]]

In [69]:
human_sr_pos

,Unnamed: 0,unnamed: 0,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
32,32,68,18222667,MEDLINE,Human,Pelargonium sidoides for acute bronchitis: a s...,"[{'header': 'OBJECTIVE', 'text': 'To criticall...",OBJECTIVE\n\n\nTo critically assess the effica...,"[{'Initials': 'TB', 'LastName': 'Agbabiaka', '...",Phytomedicine : international journal of phyto...,Phytomedicine,2008.0,"['Acute Disease', 'Bronchitis', 'Humans', 'Pel...",May,15,5,"{'page_from': '378', 'page_to': '385'}","['Journal Article', 'Meta-Analysis', ""Research...",[],['10.1016/j.phymed.2007.11.023']
35,35,81,18222681,MEDLINE,Human,Diagnostic accuracy of adenosine deaminase in ...,"[{'header': 'BACKGROUND', 'text': 'Conventiona...",BACKGROUND\n\n\nConventional tests are not alw...,"[{'Initials': 'QL', 'LastName': 'Liang', 'Fore...",Respiratory medicine,Respir Med,2008.0,"['Adenosine Deaminase', 'Biomarkers', 'Clinica...",May,102,5,"{'page_from': '744', 'page_to': '754'}","['Journal Article', 'Meta-Analysis', ""Research...",[],['10.1016/j.rmed.2007.12.007']
65,65,141,18222742,MEDLINE,Human,Evaluation of surgical procedures for sex reas...,"[{'header': 'OBJECTIVES', 'text': 'To evaluate...",OBJECTIVES\n\n\nTo evaluate earlier reviews an...,"[{'Initials': 'PA', 'LastName': 'Sutcliffe', '...","Journal of plastic, reconstructive & aesthetic...",J Plast Reconstr Aesthet Surg,2009.0,['Diagnostic and Statistical Manual of Mental ...,Mar,62,3,{},"['Journal Article', 'Review', 'Systematic Revi...",[],['10.1016/j.bjps.2007.12.009']
93,93,386,18222988,MEDLINE,Human,Systematic review of the nutritional supplemen...,"[{'header': '_UNSTRUCTURED', 'text': 'Compleme...",Complementary treatments for osteoarthritis (O...,"[{'Initials': 'S', 'LastName': 'Brien', 'ForeN...",QJM : monthly journal of the Association of Ph...,QJM,2008.0,"['Animals', 'Clinical Trials as Topic', 'Dieta...",Mar,101,3,"{'page_from': '167', 'page_to': '179'}","['Journal Article', 'Research Support, N.I.H.,...",[],['10.1093/qjmed/hcm108']
214,214,736,18223352,MEDLINE,Human,Prevention of atrial fibrillation by way of ab...,"[{'header': '_UNSTRUCTURED', 'text': 'The reni...",The renin-angiotensin-aldosterone system (RAAS...,"[{'Initials': 'MB', 'LastName': 'Jibrini', 'Fo...",American journal of therapeutics,Am J Ther,2008.0,"['Angiotensin II Type 1 Receptor Blockers', 'A...",NaN,15,1,"{'page_from': '36', 'page_to': '43'}","['Journal Article', 'Meta-Analysis', 'Review',...",[],['10.1097/MJT.0b013e31804beb59']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8584697,9240859,29777,24210209,MEDLINE,Human,Hepaticoduodenostomy versus hepaticojejunostom...,"[{'header': 'BACKGROUND', 'text': 'Excision ha...",BACKGROUND\n\n\nExcision has been established ...,"[{'Initials': 'SK', 'LastName': 'Narayanan', '...",Journal of pediatric surgery,J Pediatr Surg,2013.0,"['Anastomosis, Roux-en-Y', 'Anastomotic Leak',...",Nov,48,11,"{'page_from': '2336', 'page_to': '2342'}","['Comparative Study', 'Journal Article', 'Meta...",[],['10.1016/j.jpedsurg.2013.07.020']
8584698,9240860,29778,24210210,MEDLINE,Human,Long-term follow-up of functional outcome in p...,"[{'header': 'BACKGROUND', 'text': 'Reconstruct...",BACKGROUND\n\n\nReconstructive surgery is perf...,"[{'Initials': 'HP', 'LastName': 'Versteegh', '...",Journal of pediatric surgery,J Pediatr Surg,2013.0,"['Abnormalities, Multiple', 'Anal Canal', 'Ano...",Nov,48,11,"{'page_from': '2343', 'page_to': '2350'}","['Journal Article', 'Review', 'Systematic Revi...",[],['10.1016/j.jpedsurg.2013.08.027']
8584721,9240892,29844,24210283,MEDLINE,Human,Physical and sexual abuse in orphaned compared...,"[{'header': '_UNSTRUCTURED', 'text': 'This sys...",This systematic review assessed the quantitati...,"[{'Initials': 'J', 'LastName': 'Nichols', 'For...",Child abuse & neglect,Child Abuse Negl,2014.0,"['Africa South of the Sahara', 'Child', 'Child...",Feb,38,2,"{

In [58]:
human_sr_pos = human_sr_pos.drop(columns = ['indexing_method', 'abstract', 'month', 'volume', 'issue', 'dois', 'ptyp', 'registry_ids', 'pages', 'Unnamed: 0'])

In [60]:
human_sr_pos['label'] = 1

In [63]:
human_sr = pd.concat([human_sr_neg, human_sr_pos])

In [64]:
human_sr = human_sr.sample(frac=1, random_state=42)

In [65]:
human_sr

,pmid,status,title,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,label
9533,1901457,MEDLINE,A novel metalloproteinase present in freshly i...,We have utilized [3H] gelatin to document high...,"[{'Initials': 'Q', 'LastName': 'Le', 'ForeName...",The American journal of physiology,Am J Physiol,1991,"['Animals', 'Cell Fractionation', 'Dithiothrei...",0
5121,18249490,MEDLINE,Effect of pH on phenol biosorption by marine s...,Biosorption of phenol by the marine seaweeds L...,"[{'Initials': 'AE', 'LastName': 'Navarro', 'Fo...",Journal of hazardous materials,J Hazard Mater,2008,"['Adsorption', 'Biomass', 'Hydrogen-Ion Concen...",0
10713,1907206,MEDLINE,"[Patient oriented, prospective comparative stu...",In a prospective cross-over study involving 30...,"[{'Initials': 'JW', 'LastName': 'Brandes', 'Fo...",Biomedizinische Technik. Biomedical engineering,Biomed Tech (Berl),1991,"['Adolescent', 'Adult', 'Aged', 'Enteral Nutri...",0
14181,18237457,MEDLINE,Effects of mindful and non-mindful exercises o...,PURPOSE\n\n\nAn emerging body of evidence has ...,"[{'Initials': 'HW', 'LastName': 'Tsang', 'Fore...",The British journal of clinical psychology,Br J Clin Psychol,2008,"['Adult', 'Aged', 'Breathing Exercises', 'Cont...",1
1390,18229890,MEDLINE,Host perception and signal transduction studie...,BACKGROUND\n\n\nQuinoxyfen is a potent and eff...,"[{'Initials': 'S', 'LastName': 'Lee', 'ForeNam...",Pest management science,Pest Manag Sci,2008,"['Ascomycota', 'Carboxylic Ester Hydrolases', ...",0
...,...,...,...,...,...,...,...,...,...,...
11573,1911560,MEDLINE,Some biophysical applications of motional cont...,The principal advantage of the n.m.r. imaging ...,"[{'Initials': 'PJ', 'LastName': 'Back', 'ForeN...",International journal of biological macromolec...,Int J Biol Macromol,1991,"['Animals', 'Biophysics', 'Caseins', 'Cattle',...",0
5452,18251175,MEDLINE,Rapamycin enhances the anti-tumor effect of ge...,BACKGROUND/AIMS\n\n\nRapamycin is a potent inh...,"[{'Initials': 'T', 'LastName': 'Okada', 'ForeN...",Hepato-gastroenterology,Hepatogastroenterology,2007,"['Antibiotics, Antineoplastic', 'Blotting, Wes...",0
5652,18252223,MEDLINE,Submicroscopic duplications of the hydroxyster...,Submicroscopic copy-number imbalances contribu...,"[{'Initials': 'G', 'LastName': 'Froyen', 'Fore...",American journal of human genetics,Am J Hum Genet,2008,"['3-Hydroxyacyl CoA Dehydrogenases', 'Base Seq...",0
967,18227740,MEDLINE,Obsessive-compulsive spectrum phenomena and th...,NaN,"[{'Initials': 'E', 'LastName': 'Hollander', 'F...",CNS spectrums,CNS Spectr,2008,['Diagnostic and Statistical Manual of Mental ...,0


In [14]:
sr_pos_ids = prop_typ_id_dict['is_sr']['pos']
sr_pos_set  = pos_data_human.loc[sr_pos_ids]

In [19]:
sr_neg_ids = prop_typ_id_dict['is_sr']['neg']
sr_neg_set  = neg_data_human.loc[sr_neg_ids]

In [18]:
len(neg_data_human)

5683273

In [19]:
len(sr_neg_ids)

5659292

In [28]:
pos_data_human

,Unnamed: 0,unnamed: 0,pmid,status,indexing_method,title,abstract,abstract_plaintext,authors,journal,journal_abbrv,year,mesh,month,volume,issue,pages,ptyp,registry_ids,dois
0,0,1,18222599,MEDLINE,Human,Laparoscopic versus open partial nephrectomy: ...,"[{'header': 'OBJECTIVES', 'text': 'To critical...",OBJECTIVES\n\n\nTo critically review the curre...,"[{'Initials': 'F', 'LastName': 'Porpiglia', 'F...",European urology,Eur Urol,2008.0,"['Carcinoma, Renal Cell', 'Clinical Competence...",Apr,53,4,{},"['Journal Article', 'Review']",[],['10.1016/j.eururo.2008.01.025']
1,1,3,18222601,MEDLINE,Human,Editorial comment on: expression and prognosti...,[],NaN,"[{'Initials': 'P', 'LastName': 'Härkönen', 'Fo...",European urology,Eur Urol,2008.0,"['Annexin A3', 'Humans', 'Male', 'Prognosis', ...",Dec,54,6,"{'page_from': '1322', 'page_to': '1322'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.002']
2,2,4,18222602,MEDLINE,Human,Editorial comment on: celecoxib has potent ant...,[],NaN,"[{'Initials': 'S', 'LastName': 'Brandau', 'For...",European urology,Eur Urol,2008.0,"['Animals', 'BCG Vaccine', 'Carcinoma, Transit...",Sep,54,3,"{'page_from': '629', 'page_to': '630'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.014']
3,3,5,18222603,MEDLINE,Human,Editorial comment on: expression and prognosti...,[],NaN,"[{'Initials': 'GJ', 'LastName': 'van Leenders'...",European urology,Eur Urol,2008.0,"['Annexin A3', 'Humans', 'Male', 'Microarray A...",Dec,54,6,"{'page_from': '1323', 'page_to': '1323'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.003']
4,4,6,18222604,MEDLINE,Human,Considerations on the use of urine markers for...,[],NaN,"[{'Initials': 'BW', 'LastName': 'van Rhijn', '...",European urology,Eur Urol,2008.0,"['Biomarkers, Tumor', 'Carcinoma, Transitional...",May,53,5,"{'page_from': '880', 'page_to': '881'}","['Comment', 'Editorial']",[],['10.1016/j.eururo.2008.01.043']
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8584786,8584786,29982,24210460,MEDLINE,Human,Pregnancy-related knowledge and information ne...,"[{'header': '_UNSTRUCTURED', 'text': ""For wome...","For women with epilepsy (WWE), pregnancy is co...","[{'Initials': 'A', 'LastName': 'McGrath', 'For...",Epilepsy & behavior : E&B,Epilepsy Behav,2014.0,"['Anticonvulsants', 'Epilepsy', 'Female', 'Hea...",Feb,31,NaN,"{'page_from': '246', 'page_to': '255'}","['Journal Article', ""Research Support, Non-U.S...",[],['10.1016/j.yebeh.2013.09.044']
8584787,8584787,29984,24210462,MEDLINE,Human,Australia's seizure divide - indigenous versus...,"[{'header': '_UNSTRUCTURED', 'text': 'Indigeno...",Indigenous Australians suffer the highest mort...,"[{'Initials': 'C', 'LastName': 'Plummer', 'For...",Epilepsy & behavior : E&B,Epilepsy Behav,2014.0,"['Adolescent', 'Adult', 'Australia', 'Ethnic G...",Feb,31,NaN,"{'page_from': '363', 'page_to': '368'}","['Comparative Study', 'Journal Article', ""Rese...",[],['10.1016/j.yebeh.2013.09.042']
8584788,8584788,29986,24210464,MEDLINE,Human,Research on psychosocial aspects of epilepsy i...,"[{'header': '_UNSTRUCTURED', 'text': 'This stu...",This study reviewed research conducted on the ...,"[{'Initials': 'JM', 'LastName': 'Al-Khateeb', ...",Epilepsy & behavior : E&B,Epilepsy Behav,2014.0,"['Biomedical Research', 'Epilepsy', 'Health Kn...",Feb,31,NaN,"{'page_from': '256', 'page_to': '262'}","['Journal Article', 'Review']",[],['10.1016/j.yebeh.2013.09.033']
8584789,8584789,29989,24210468,MEDLINE,Human,Emerging methods for the fabrication of polyme...,"[{'header': '_UNSTRUCTURED', 'text': 'Hollow p...",Hollow polymer capsules are attracting increas...,"[{'Initials': 'J', 'LastName': 'Cui', 'ForeNam...",Advances in colloid and interface science,Adv Colloid Interface Sci,2014.0,"['Capsules', 'Chemical Phenomena', 'Drug Compo...",May,207,NaN,"{'page_from': '14', 'page_to': '31'}","['Journal Article', ""Research Support, Non-U.S...",[],['10.1016/j.cis.2013.10.012']
